# Final Part 3

# Section 1: Review of Part 2 

Review of 2:

My system models cooling of a turbine blade using a forced convection ODE. The temperature changes based on heat loss to surrounding air, which varies slowly over time. In Part 2, the problem was solved using the Backward Euler method. Because this method is implicit, each time step required solving an equation using Newton-Raphson iteration. The system is stiff because it has two very different time scales, First is has a very fast cooling rate due to a large convection coefficient, and secondly a much slower change in the ambient temperature. This large separation causes explicit methods like Euler Forward to become unstable unless very small time steps are used, which motivates using implicit methods instead.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
#Brief recap of your problem and basic implicit solver
#State the stiffness parameter and why the problem is stiff
#PUT THE IMPLICIT SOLVER

#Parameters:
conv_h = 1500
t_start = 0
t_end = 10
T_initial = 1000
dt = 0.01

#ambeint forcing
def T_inf(t):
    return 300 + 10*np.sin(0.01*t)
#Guiding ODE
def dTdt(T, t):
    return -conv_h * (T - T_inf(t))

#stiffness stuff
tau_fast = 1 / conv_h
tau_slow = 1 / 0.01 # forcing time scale
stiffness_ratio = tau_slow / tau_fast
print("Fast time scale:", tau_fast)
print("Slow time scale:", tau_slow)
print("Stiffness ratio:", stiffness_ratio)

#write the residual function for specific problem
def residual(T_new, T_old, t_new, dt):
    return (1 + conv_h*dt)*T_new - T_old - conv_h*dt*T_inf(t_new)
#analytically derive g prime
def residual_derivative(dt):
    return 1 + conv_h*dt

In [ ]:
#Backward Eulers with Newton Raphson:
#newton_raphson function
def newton_raphson(F, F_prime, x0, tol=1e-8, max_iter=10):
    x = x0
    for k in range(max_iter):
        x_new = x - F(x) / F_prime(x)
        if abs(x_new - x) < tol:
            return x_new, k+1, True
        x = x_new
    return x, max_iter, False

def backward_euler_step(T_old, t_new, dt):

    # implicit equation F(T_{n+1}) = 0
    def F(T):
        return T - T_old - dt * (-conv_h * (T - T_inf(t_new)))

    # derivative (constant in your linear case)
    def F_prime(T):
        return 1 + conv_h * dt

    # initial guess
    T0 = T_old

    # solve using Newton
    T_new, iters, ok = newton_raphson(F, F_prime, T0)

    return T_new, iters, ok

t_values = np.arange(t_start, t_end + dt, dt)
T_values = np.zeros(len(t_values))
iters_list = []

T_values[0] = T_initial

for i in range(1, len(t_values)):
    T_values[i], iters, _ = backward_euler_step(
        T_values[i-1], t_values[i], dt
    )
    iters_list.append(iters)

# Section 2: Newton-Raphson Damping Implementation

#explaining damping and line search

Damping improves the stability of a function by scaling the Newton Raphson by a factor between 0 and 1. In doing this the damping method can check whether the residual is reduced or not. If it's found that the residual is not reduced, the value of damping gets halfed until theres a safe enough step that reduces the residual. Essentially, it checks that newton raphson is not overshooting the solution.

Line Search is the method that the damping coefficient uses. Essentially, line searching works by moving along the newton direction for a step size that guarentees improvement. The goal being that each step size reduces the residual to prevent divergence and converge to the solution.

In [ ]:
def damped_newton_raphson(y_n, t_next, h, g_func, g_prime_func, tol=1e-8, max_iter=20, alpha_min=1e-4):
    y_guess = y_n
    for k in range(max_iter):
        g = g_func(y_guess, y_n, t_next, h)
        if abs(g) < tol:
            return y_guess, k, True
        g_prime = g_prime_func(h)
        delta = -g / g_prime
        # Line search for damping factor
        alpha = 1.0
        g_current = abs(g)
        for _ in range(10):  # Max 10 backtracking steps
            y_new = y_guess + alpha * delta
            g_new = abs(g_func(y_new, y_n, t_next, h))
            if g_new < g_current:  # Sufficient decrease
                break
            alpha = alpha / 2.0
            if alpha < alpha_min:
                print(f"Warning: alpha too small at iter {k}")
                return y_guess, k, False
        y_guess = y_new
    return y_guess, max_iter, False


In [ ]:
#Demonstration where damping helps (e.g., very large initial h)
# large step size test
h_test = 2.0
t_test = 1.0
T_old = 1000

# basic NR
y_basic, it_basic, ok_basic = backward_euler_step(T_old, t_test, h_test) #running my old newton raphson from part 2

# damped NR
y_damped, it_damped, ok_damped = damped_newton_raphson(T_old, t_test, h_test,residual, residual_derivative)

print("Basic Newton Raphson:")
print("Converged:", ok_basic)
print("Iterations:", it_basic)
print("Result:", y_basic)

print("\nDamped Newton Raphson:")
print("Converged:", ok_damped)
print("Iterations:", it_damped)
print("Result:", y_damped)

#Comparing basic NR to damped NR convergence

There is an issue in quantifying how damped NR convergence would help with my system of equations. Since my system is linear, the regular newton raphson will converge in one to two iterations, due to the linearity of my system. This shows that regardless of if i use Damped Newton Raphson or Basic Newton Raphson it will still converge in 1 to 2 iterations no matter how big the h step is. However, despite this the damped newton was still more efficient, converging in one less step than the regular newton raphson

# Section 3: Adaptive Step Size Implementation

#Explanation of step-doubling error estimation

Step double error estimation works by comparing a single big step to two smaller half steps. Fixed step size can be inadquate for balancing both cost efficiency and accuracy. For example when an h size is too large the method may fail to capture the accuracy of the desired function. However, when the h size is too small the method will conduct unnecessary computations that cause an increased computational cost for marginal accuracy improvements. However, the step double approach makes the h adaptive to balance both computational cost and accuracy by using the big step and two half steps to estimate which has a lower error while not sacrificing unnecessary computational time.


In [5]:
#Implementation with clear comments showing:
#Single step of size h
#Two half-steps of size h/2
#Error estimation and step size adjustment

def estimate_error_step_doubling(T_n, t_n, h):
    # ONE full step using backward euler
    T_full, it1, _ = backward_euler_step(T_n, t_n + h, h)
    # TWO half steps using backward euler
    T_half_1, it2, _ = backward_euler_step(T_n, t_n + h/2, h/2)
    T_half_2, it3, _ = backward_euler_step(T_half_1, t_n + h, h/2)
    # error estimate for steps
    err = abs(T_half_2 - T_full)
    # total cost (for analysis later)
    total_iters = it1 + it2 + it3
    return T_full, T_half_2, err, total_iters

#computing new step size:
def adaptive_step(T_n, t_n, h, tol):
    eps = 1e-10
    safety = 0.9
    h_min = 1e-6
    h_max = 1.0
    T_full, T_half, err, iters = estimate_error_step_doubling(T_n, t_n, h)
    # compute new step size
    if err < 1e-14:
        h_new = min(2*h, h_max)
    else:
        h_new = h * np.sqrt(tol / (err + eps))
        h_new = safety * h_new
        h_new = max(h_min, min(h_max, h_new))
    # prevent extreme jumps (slightly smoother)
    h_new = max(0.5*h, min(2.0*h, h_new))
    # accept/reject
    if err < tol:
        return T_half, h, h_new, True, iters, err   # ACCEPT
    else:
        return T_n, h, h_new, False, iters, err    # REJECT

#which step to use
def adaptive_solver(T0, t0, t_end, h0, tol):
    t_vals = [t0]
    T_vals = [T0]
    h_vals = []
    err_vals = []   # plots error for later
    t = t0
    T = T0
    h = h0
    accepted = 0
    rejected = 0
    total_iters = 0
    while t < t_end:
        # prevent overshooting final time
        if t + h > t_end:
            h = t_end - t
        T_new, h_used, h_new, ok, iters, err = adaptive_step(T, t, h, tol)
        total_iters += iters
        if ok:
            # accept step
            t += h_used
            T = T_new
            t_vals.append(t)
            T_vals.append(T)
            h_vals.append(h_used)
            err_vals.append(err)   #tracks error
            h = h_new
            accepted += 1
        else:
            # reject step → retry with smaller h
            h = h_new
            rejected += 1
    stats = {"accepted": accepted, "rejected": rejected, "total_iters": total_iters}

    return np.array(t_vals), np.array(T_vals), np.array(h_vals), np.array(err_vals), stats

In [ ]:
#Visualization of how h adapts during solution
#running the adaptive solver:
t_adapt, T_adapt, h_hist, err_hist, stats = adaptive_solver(T_initial, t_start, t_end, h0=0.1, tol=1e-4)
print("Accepted steps:", stats["accepted"])
print("Rejected steps:", stats["rejected"])
print("Total NR iterations:", stats["total_iters"])

#Plotting the adaptive solver:
#solution:
plt.figure()
plt.plot(t_adapt, T_adapt, '-o', markersize=3)
plt.title("Adaptive Solution")
plt.xlabel("Time")
plt.ylabel("Temperature")
plt.grid()
plt.show()

#step size graph:
plt.figure()
plt.plot(t_adapt[:-1], h_hist, '-o')
plt.title("Step Size vs Time")
plt.xlabel("Time")
plt.ylabel("h")
plt.grid()
plt.show()

#error plot
plt.figure()
plt.figure()
plt.semilogy(t_adapt[1:], err_hist, '-o', markersize=3)
plt.axhline(y=1e-4, linestyle='--',label="tolerance")
plt.title("Error Estimate vs Time")
plt.xlabel("Time")
plt.ylabel("Error")
plt.grid()
plt.show()

# Section 4: Performance Analysis

In [ ]:
#Fixed vs. Adaptive comparison:
#Same target accuracy: compare total steps, function evaluations
#Same step budget: compare final accuracy

#FIXED STEP BACKWARD EULER SOLUTION
t_fixed = np.arange(t_start, t_end + dt, dt)
T_fixed = np.zeros(len(t_fixed))
T_fixed[0] = T_initial

fixed_iters = 0

for i in range(1, len(t_fixed)):
    T_fixed[i], iters, _ = backward_euler_step(
        T_fixed[i-1], t_fixed[i], dt
    )
    fixed_iters += iters

#ADAPTIVE SOLUTION (already computed in Section 3)
# t_adapt, T_adapt, h_hist, err_hist, stats already exist


# ============================================================
# 3. COMPUTATIONAL COST COMPARISON
# ============================================================
adaptive_iters = stats["total_iters"]

print("=== COMPUTATIONAL COST ===")
print("Fixed Backward Euler:")
print("  Steps:", len(t_fixed))
print("  Newton iterations:", fixed_iters)

print("\nAdaptive Backward Euler:")
print("  Steps:", len(t_adapt))
print("  Accepted:", stats["accepted"])
print("  Rejected:", stats["rejected"])
print("  Newton iterations:", adaptive_iters)


# ============================================================
# 4. ACCURACY COMPARISON (FINAL VALUE ERROR)
# ============================================================
final_error = abs(T_fixed[-1] - T_adapt[-1])

print("\n=== ACCURACY COMPARISON ===")
print("Final fixed T:", T_fixed[-1])
print("Final adaptive T:", T_adapt[-1])
print("Absolute difference:", final_error)


In [ ]:
#Plots:
plt.figure()
plt.plot(t_fixed, T_fixed, label="Fixed Backward Euler")
plt.plot(t_adapt, T_adapt, '-o', markersize=3, label="Adaptive Backward Euler")
plt.title("Solution Comparison")
plt.xlabel("Time")
plt.ylabel("Temperature")
plt.legend()
plt.grid()
plt.show()

# --- Adaptive step size history ---
plt.figure()
plt.plot(t_adapt[:-1], h_hist, '-o')
plt.title("Adaptive Step Size h(t)")
plt.xlabel("Time")
plt.ylabel("h")
plt.grid()
plt.show()

# --- Adaptive error evolution ---
plt.figure()
plt.semilogy(t_adapt[1:], err_hist, '-o')
plt.axhline(y=1e-4, linestyle='--', label="Tolerance")
plt.title("Adaptive Error Estimate")
plt.xlabel("Time")
plt.ylabel("Error")
plt.legend()
plt.grid()
plt.show()

#Solution with adaptive time points marked
#Step size h(t) vs. time
#Error estimate vs. time
#Accepted vs. rejected steps
#Comparison table (fixed vs adaptive)

In [ ]:
#summary of information presented:
print("\n=== SUMMARY TABLE ===")
print(f"{'Method':<15}{'Steps':<10}{'Iterations':<15}")
print("-"*40)
print(f"{'Fixed':<15}{len(t_fixed):<10}{fixed_iters:<15}")
print(f"{'Adaptive':<15}{len(t_adapt):<10}{adaptive_iters:<15}")

# Section 5: Robustness Testing

In [ ]:
#Test with very large initial h (e.g., h₀ = 1.0)

In [ ]:
#Test with very tight tolerance (e.g., tol = 10⁻⁸)

In [ ]:
#Show that adaptive solver handles both gracefully

# Section 6: Conclusions

In [ ]:
#When is damping necessary?
#When are adaptive step sizes worth the complexity?
#Key lessons learned